# Step 3b — Appendix to the raw data-quality audit: checks that notebook 03 did not make

**Purpose.** Notebook 03 audited the anime and user tables and the counts of the list table. When we read it again as a whole, we found places where we looked at one table or one kind of column and not at the others, or where we described a column without testing it. This appendix runs those missing checks, so that the audit has no blind spot before the cleaning decisions of step 4. It changes nothing in the data. After the run, its cells are merged into notebook 03.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Part A only reads the two small tables and takes seconds. Part B reads the 80M-row list table in one pass (a few minutes, 3M rows at a time); set `RUN_LISTS = False` in the setup cell to skip it. The pass and its report are two separate cells, so a mistake in the report does not need a second pass.

**What was missing, and why it matters.**

| # | Gap in notebook 03 | Why it matters | Where |
|---|---|---|---|
| 1 | The list table (80M rows, the largest table) was never scanned for placeholders. The first look (1a) skipped it on purpose because of its size. | A placeholder in the largest table affects the most rows. `my_tags` is free text, and the two date columns can hold default or impossible dates | B1 to B3 |
| 2 | Numeric columns were never scanned for placeholder numbers (`-1`, `999`, `9999`). The first look (1a) read text columns only. | A number such as `-1` or `9999` standing for "unknown" would enter every mean and correlation | A1 |
| 3 | The usernames were checked for being unique (exact match), not for **capital letters and spaces**. | The username is the join key between `UserList` and `UserAnimeList`. Two spellings of one name, or a trailing space, break a join without any error | A2, B4 |
| 4 | `location` was only counted by its most frequent values. | If `USA`, `usa` and `Usa ` are three values, every grouping by country is split | A3 |
| 5 | Duplicates were checked for `anime_id` only. | The same anime may exist under two ids, which would double its weight | A4 |
| 6 | No check that two columns of the same row agree (`scored_by` against `members`, end date against start date, a finished anime that starts after the crawl). | A row that contradicts itself is wrong in at least one column | A5, B3 |
| 7 | The columns that hold Python text (`aired`, `related`, `opening_theme`, `ending_theme`) were read with `ast.literal_eval` in notebook 02 without counting the failures. | A parse failure silently drops a row from any analysis of these columns | A6 |
| 8 | The dates of the list table were seen as a top-5 list in notebook 02, and never tested for partial dates, impossible dates, dates after the crawl, or a finish date before the start date. `my_last_updated` was only used for its maximum. | A date range decides which rows can be used for a time analysis | B3 |
| 9 | `anime_id` of the list table was never checked against `AnimeList`, and `my_watched_episodes` was never compared with the episode count. | An id without an anime cannot be joined. More episodes watched than exist is impossible | B4, B5 |

**Evidence tags.** Everything printed is a count from the files ([V]). Explanations of *why* a value exists are hypotheses and are labelled so.


In [1]:
from collections import Counter
from pathlib import Path
import ast, html, re, sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part B, which reads 80M rows

na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
CRAWL_END = pd.Timestamp("2018-05-25", tz="UTC")  # estimated end of the crawl (notebook 02)

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("anime:", anime.shape, "| users:", users.shape, "| CSV files:", len(paths))


Python: 3.13.18 | pandas: 2.3.3
anime: (14478, 31) | users: (302675, 17) | CSV files: 9


## Part A. The two small tables

### A1. Numeric columns: impossible values and placeholder numbers

For every numeric column (the id columns are skipped) the cell prints how many cells are missing, the minimum, the maximum, how many values are negative, the five most frequent values, and how many cells equal a number that is often used as a placeholder (`-1`, `99`, `999`, `9999`, `99999`). A value such as `999` can also be real (for example a count of favorites), so a hit is a reason to look, not a proof.


In [2]:
# A1. Numeric columns: impossible values, most frequent values, placeholder-like numbers
SENTINELS = [-1, 99, 999, 9999, 99999]

def numeric_overview(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(include="number").columns:
        if col.endswith("_id"):
            continue
        s = df[col].dropna()
        top = " | ".join(f"{v:,.6g} x{c:,}" for v, c in s.value_counts().head(n).items())
        sentinel = {v: int((s == v).sum()) for v in SENTINELS if (s == v).any()}
        extra = f" | placeholder-like: {sentinel}" if sentinel else ""
        print(f"{col}: missing {int(df[col].isna().sum()):,} | min {s.min():,.6g} | max {s.max():,.6g}"
              f" | negative {int((s < 0).sum()):,} | top: {top}{extra}")

numeric_overview(anime, "anime table")
print()
numeric_overview(users, "user table")


===== anime table
episodes: missing 0 | min 0 | max 1,818 | negative 0 | top: 1 x6,857 | 2 x1,187 | 12 x1,034 | 13 x637 | 3 x558 | placeholder-like: {99: 2}
score: missing 0 | min 0 | max 10 | negative 0 | top: 0 x354 | 6 x120 | 5 x100 | 6.5 x90 | 6.33 x87
scored_by: missing 0 | min 0 | max 1.00948e+06 | negative 0 | top: 0 x308 | 8 x155 | 7 x143 | 5 x140 | 9 x139 | placeholder-like: {99: 11, 999: 2}
rank: missing 1,574 | min 0 | max 12,919 | negative 0 | top: 0 x9 | 6,303 x5 | 3,012 x4 | 12,630 x4 | 4,700 x4 | placeholder-like: {99: 1, 999: 1, 9999: 1}
popularity: missing 0 | min 0 | max 14,487 | negative 0 | top: 0 x9 | 3,252 x4 | 6,104 x4 | 2,870 x4 | 9,537 x4 | placeholder-like: {99: 1, 999: 1}
members: missing 0 | min 0 | max 1.45638e+06 | negative 0 | top: 123 x35 | 87 x35 | 85 x32 | 115 x32 | 81 x30 | placeholder-like: {99: 17}
favorites: missing 0 | min 0 | max 106,895 | negative 0 | top: 0 x5,018 | 1 x1,539 | 2 x817 | 3 x577 | 4 x407 | placeholder-like: {99: 6, 999: 1}

===== 

### A2. Spaces, capital letters and the username (the join key)

The first cell counts, in every text column of both tables, the cells with a space at the start or at the end, and the cells with two or more spaces in a row. The second one tests the usernames: are there names that differ only by capital letters or by a space? On MyAnimeList a profile address does not depend on capital letters, so two such names could be one person; and a join on the exact text would then miss it.


In [3]:
# A2. Spaces in text cells, and usernames that differ only by capital letters
def spacing(df, label):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        edge = int((s != s.str.strip()).sum())
        double = int(s.str.contains(r"\s{2,}", regex=True).sum())
        if edge or double:
            print(f"{col}: space at the start or the end {edge:,} | two or more spaces in a row {double:,}")

spacing(anime, "anime table")
print()
spacing(users, "user table")

name = users["username"]
low = name.str.lower()
clash = low.duplicated(keep=False)
print()
print("usernames:", f"{len(name):,}", "| distinct:", f"{name.nunique():,}", "| distinct after lowercase:", f"{low.nunique():,}")
print("names that differ only by capital letters:", int(clash.sum()), "in", low[clash].nunique(), "groups")
print(users.loc[clash, ["username", "user_id"]].sort_values("username", key=lambda s: s.str.lower()).head(10).to_string(index=False))
print("usernames with a space at the start or the end:", int((name != name.str.strip()).sum()))


===== anime table
title: space at the start or the end 0 | two or more spaces in a row 3
title_japanese: space at the start or the end 4 | two or more spaces in a row 32
title_synonyms: space at the start or the end 0 | two or more spaces in a row 5
background: space at the start or the end 105 | two or more spaces in a row 29
related: space at the start or the end 0 | two or more spaces in a row 2
opening_theme: space at the start or the end 0 | two or more spaces in a row 40
ending_theme: space at the start or the end 0 | two or more spaces in a row 82

===== user table
location: space at the start or the end 4,560 | two or more spaces in a row 406

usernames: 302,675 | distinct: 302,675 | distinct after lowercase: 302,675
names that differ only by capital letters: 0 in 0 groups
Empty DataFrame
Columns: [username, user_id]
Index: []
usernames with a space at the start or the end: 0


### A3. `location`: how many spellings hide behind the same place?

`location` is free text. The cell compares the number of distinct raw texts with the number after we remove capital letters and extra spaces, and prints the places that have the most spellings. This tells how much grouping work step 4 has before any analysis by country or city.


In [4]:
# A3. location: raw texts against texts without capital letters and extra spaces
loc = users["location"].dropna()
norm = loc.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("filled locations:", f"{len(loc):,}", "| distinct raw texts:", f"{loc.nunique():,}", "| distinct after normalising:", f"{norm.nunique():,}")
print("most frequent after normalising:", norm.value_counts().head(10).to_dict())

spellings = loc.groupby(norm).nunique().sort_values(ascending=False)
print("\nplaces written in the most different ways:")
for key in spellings.head(5).index:
    forms = loc[norm == key].value_counts().head(6).to_dict()
    print(f"  {key!r}: {int(spellings[key])} spellings, for example {forms}")


filled locations: 156,799 | distinct raw texts: 55,283 | distinct after normalising: 49,616
most frequent after normalising: {'poland': 2544, 'germany': 2004, 'canada': 1946, 'california': 1679, 'brazil': 1517, 'philippines': 1281, 'france': 1204, 'australia': 1101, 'sweden': 1080, 'singapore': 952}

places written in the most different ways:
  'saudi arabia': 12 spellings, for example {'Saudi Arabia': 82, 'saudi arabia': 19, 'Saudi Arabia ': 12, ' Saudi Arabia': 5, 'saudi arabia ': 3, 'Saudi arabia': 2}
  'new york': 12 spellings, for example {'New York': 661, 'new york': 45, 'New York ': 24, 'New york': 15, ' New York': 4, 'NEW YORK': 3}
  'los angeles, california': 11 spellings, for example {'Los Angeles, California': 325, 'Los Angeles, California ': 16, 'los angeles, california': 7, 'Los angeles, California': 3, 'los angeles, california ': 1, 'Los Angeles, California     ': 1}
  'somewhere over the rainbow': 11 spellings, for example {'Somewhere over the rainbow': 31, 'somewhere ov

### A4. Duplicates beyond `anime_id`

Notebook 03 only checked that `anime_id` is unique. Here we look for the same title under several ids, and for rows that are identical in every column except `anime_id` (the same anime entered twice).


In [5]:
# A4. Duplicate anime that have different ids
dup_title = anime[anime["title"].duplicated(keep=False)]
print("titles used by more than one anime_id:", dup_title["title"].nunique(), "| rows:", len(dup_title))
print(dup_title.sort_values("title")[["anime_id", "title", "type", "status", "scored_by"]].head(10).to_string(index=False))

same_but_id = anime.drop(columns=["anime_id"]).duplicated(keep=False)
print("\nrows identical in every column except anime_id:", int(same_but_id.sum()))

title_clean = anime["title"].str.strip().str.lower()
print("titles that differ only by capital letters or spaces:", int(title_clean.duplicated(keep=False).sum() - anime["title"].duplicated(keep=False).sum()))


titles used by more than one anime_id: 1 | rows: 2
 anime_id       title type           status  scored_by
    35102 Hinamatsuri  OVA  Finished Airing          8
    36296 Hinamatsuri   TV Currently Airing      12470

rows identical in every column except anime_id: 0
titles that differ only by capital letters or spaces: 0


### A5. Do the columns of one row agree with each other?

A row that contradicts itself is wrong in at least one column. The cell counts: a `score` outside 0 to 10; more raters than members (`scored_by` > `members`); more favorites than members; an end date before the start date in `aired`; a **finished** anime whose start date lies after the crawl; and a **currently airing** anime whose end date lies before the crawl (the crawl ended about 2018-05-25, see notebook 02).


In [6]:
# A5. Contradictions inside one row of the anime table
def aired_part(t, key):
    try:
        return pd.to_datetime(ast.literal_eval(t).get(key), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

air_start = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "from")), utc=True)
air_end = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "to")), utc=True)

print("score outside 0 to 10:", int(((anime["score"] < 0) | (anime["score"] > 10)).sum()))
print("scored_by > members:", int((anime["scored_by"] > anime["members"]).sum()),
      "| favorites > members:", int((anime["favorites"] > anime["members"]).sum()))
print("end date before start date:", int((air_end < air_start).sum()))
finished_late = (anime["status"] == "Finished Airing") & (air_start > CRAWL_END)
airing_over = (anime["status"] == "Currently Airing") & (air_end < CRAWL_END)
print("Finished Airing with a start date after the crawl:", int(finished_late.sum()))
print("Currently Airing with an end date before the crawl:", int(airing_over.sum()))
cols = ["anime_id", "title", "status", "aired_string"]
print(anime.loc[finished_late, cols].head(5).to_string(index=False))
print(anime.loc[airing_over, cols].head(5).to_string(index=False))


score outside 0 to 10: 0
scored_by > members: 0 | favorites > members: 1
end date before start date: 32
Finished Airing with a start date after the crawl: 16
Currently Airing with an end date before the crawl: 6
 anime_id                    title          status              aired_string
    21277            Yawaraka Atom Finished Airing    Oct, 2007 to Oct, 2008
    33756               Hen Xi You Finished Airing Jun, 2016 to Nov 28, 2016
    13819         Kamiusagi Rope 2 Finished Airing    Nov, 2010 to Dec, 2011
    19695               Clover 4/3 Finished Airing    Aug, 2000 to Dec, 2000
    28733 Ginga Tetsudou 999 (ONA) Finished Airing    Sep, 2002 to Jul, 2004
 anime_id             title           status                 aired_string
    36623        SNS Police Currently Airing Mar 24, 2018 to May 19, 2018
    35857 Lupin III: Part 5 Currently Airing          Apr 4, 2018 to 2018
    36902  Mahou Shoujo Ore Currently Airing          Apr 2, 2018 to 2018
    34572      Black Clover Cu

### A6. The columns that hold Python text: how many cells cannot be read?

`aired`, `related`, `opening_theme` and `ending_theme` are texts written like a Python dictionary or list. Notebook 02 read them with `ast.literal_eval`, and a cell that fails was silently turned into a missing value. Here the failures are counted.


In [7]:
# A6. Python-literal columns: how many cells fail to parse?
for col in ["aired", "related", "opening_theme", "ending_theme"]:
    s = anime[col].dropna()
    bad = []
    for v in s:
        try:
            ast.literal_eval(v)
        except Exception:
            bad.append(v)
    example = bad[0][:80] if bad else "-"
    print(f"{col:14s} cells: {len(s):>6,} | cannot be parsed: {len(bad):>5,} | example: {example!r}")


aired          cells: 14,478 | cannot be parsed:     0 | example: '-'
related        cells: 14,478 | cannot be parsed:     0 | example: '-'
opening_theme  cells: 14,478 | cannot be parsed:     0 | example: '-'
ending_theme   cells: 14,478 | cannot be parsed:     0 | example: '-'


### Observations for part A

**TODO (fill in after the run):**
- A1: which numeric columns have negative values, or a placeholder-like number? Do the most frequent values of any column look like a placeholder (for example `0` or `-1`)?
- A2: which text columns have spaces at the start or at the end? How many usernames differ only by capital letters or by a space, and can the pairs be joined to the same person?
- A3: how many raw spellings of `location` collapse after normalising? Which places have the most spellings?
- A4: are there titles shared by several ids, and are some of them true duplicates (rows identical except `anime_id`)?
- A5: how many rows contradict themselves, and which check finds the most?
- A6: does any of the four columns have cells that cannot be parsed?


## Part B. The list table (80M rows), read once

The list table was never scanned for placeholders (the first look of notebook 03 skipped it because of its size). It is read once, 3M rows at a time, and each chunk only updates counters. A text column with a few thousand distinct values (the two dates) is counted exactly; `my_tags`, which has far too many distinct values, is counted by pattern and by an approximate top list (the 300 most frequent values of each chunk are merged, so a frequent value is found and its count is a lower bound).

| Cell | What it does |
|---|---|
| B1 | The one pass. Prints only its progress |
| B2 | Report on `my_tags` |
| B3 | Report on the two date columns and on `my_last_updated` |
| B4 | Report on `anime_id` and on the usernames of the list table |
| B5 | Report on `my_watched_episodes` against the episode count |


In [8]:
# B1. One pass over the list table: only counters are updated, the reports are in the next cells
if RUN_LISTS:
    CHUNK = 3_000_000
    DATE_OK = r"^\d{4}-\d{2}-\d{2}$"
    REAL_DATE = r"^(?!0000)\d{4}-(?!00)\d{2}-(?!00)\d{2}$"
    CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
    EPOCH_SITE = pd.Timestamp("2004-11-01", tz="UTC").timestamp()   # MyAnimeList did not exist before
    EPOCH_END = CRAWL_END.timestamp() + 86400                       # end of 2018-05-25

    use = ["username", "anime_id", "my_watched_episodes", "my_status", "my_start_date",
           "my_finish_date", "my_last_updated", "my_tags"]
    dtypes = {"anime_id": "int32", "my_watched_episodes": "float32", "my_status": "int8", "my_last_updated": "float64"}
    episodes_of = anime.set_index("anime_id")["episodes"]
    known_ids = set(anime["anime_id"])

    rows = empty_username = orphan_rows = 0
    names, ids = set(), set()
    date_counts = {"my_start_date": Counter(), "my_finish_date": Counter()}
    both_real = finish_before_start = 0
    tag_filled = tag_html = 0
    tag_placeholder, tag_top = Counter(), Counter()
    upd_min, upd_max = float("inf"), float("-inf")
    upd_nonpositive = upd_before_site = upd_after_crawl = 0
    ep_known = watched_over = completed_rows = completed_zero = completed_less = 0

    reader = pd.read_csv(paths["UserAnimeList.csv"], usecols=use, dtype=dtypes, chunksize=CHUNK, **na_kwargs)
    for k, chunk in enumerate(reader, start=1):
        rows += len(chunk)
        empty_username += int(chunk["username"].isna().sum())
        names.update(chunk["username"].dropna().unique().tolist())
        ids.update(chunk["anime_id"].unique().tolist())
        orphan_rows += int((~chunk["anime_id"].isin(known_ids)).sum())

        for col in date_counts:
            date_counts[col].update(chunk[col].fillna("<empty>").value_counts().to_dict())
        st = chunk["my_start_date"].fillna("")
        fi = chunk["my_finish_date"].fillna("")
        real = st.str.match(REAL_DATE) & fi.str.match(REAL_DATE)
        both_real += int(real.sum())
        finish_before_start += int((real & (fi < st)).sum())

        tags = chunk["my_tags"].dropna()
        tag_filled += len(tags)
        if len(tags):
            tag_html += int(tags.str.contains(r"&#?\w+;", regex=True).sum())
            is_placeholder = tags.str.match(CANDIDATE, flags=re.IGNORECASE)
            tag_placeholder.update(tags[is_placeholder].str.lower().value_counts().to_dict())
            tag_top.update(tags.value_counts().head(300).to_dict())

        upd = chunk["my_last_updated"].dropna()
        if len(upd):
            upd_min, upd_max = min(upd_min, float(upd.min())), max(upd_max, float(upd.max()))
            upd_nonpositive += int((upd <= 0).sum())
            upd_before_site += int((upd < EPOCH_SITE).sum())
            upd_after_crawl += int((upd > EPOCH_END).sum())

        watched = chunk["my_watched_episodes"]
        total_eps = chunk["anime_id"].map(episodes_of)
        has_eps = total_eps > 0
        done = chunk["my_status"] == 2
        ep_known += int(has_eps.sum())
        watched_over += int((has_eps & (watched > total_eps)).sum())
        completed_rows += int(done.sum())
        completed_zero += int((done & (watched == 0)).sum())
        completed_less += int((done & has_eps & (watched < total_eps)).sum())

        if k % 5 == 0:
            print(f"  ... {k} chunks, {rows:,} rows")
    print(f"done: {rows:,} rows in {k} chunks")
else:
    print("Skipped (RUN_LISTS = False)")


  ... 5 chunks, 15,000,000 rows
  ... 10 chunks, 30,000,000 rows
  ... 15 chunks, 45,000,000 rows
  ... 20 chunks, 60,000,000 rows
  ... 25 chunks, 75,000,000 rows
done: 80,076,112 rows in 27 chunks


In [9]:
# B2. my_tags: how many are filled, which are placeholder-like, which are the most frequent
if RUN_LISTS:
    print(f"my_tags filled: {tag_filled:,} of {rows:,} rows ({tag_filled / rows:.1%}) | with an HTML code: {tag_html:,}")
    print("placeholder-like tags:", f"{sum(tag_placeholder.values()):,}", dict(tag_placeholder.most_common(8)))
    print("most frequent tags (approximate, a lower bound):")
    for v, c in tag_top.most_common(10):
        print(f"   {v[:40]!r} {c:,}")
else:
    print("Skipped (RUN_LISTS = False)")


my_tags filled: 4,760,289 of 80,076,112 rows (5.9%) | with an HTML code: 8,758
placeholder-like tags: 53,395 {'undefined': 33194, '-': 6194, 'null': 1957, '.': 1423, '?': 1422, '...': 1162, '---': 1002, '???': 694}
most frequent tags (approximate, a lower bound):
   'undefined' 33,172
   'Hentai' 31,295
   'Sub' 14,664
   'hentai' 14,142
   'Comedy' 13,664
   'anYme' 13,480
   'comedy' 12,967
   '*' 9,237
   'Action' 8,116
   'Dub' 7,825


In [10]:
# B3. The two date columns and my_last_updated
if RUN_LISTS:
    def summarize_dates(counter, label):
        s = pd.Series(counter, dtype="int64")
        total = int(s.sum())
        txt = pd.Series(s.index, index=s.index)
        is_format = txt.str.match(DATE_OK)
        is_empty = txt == "<empty>"
        is_zero = txt == "0000-00-00"
        other = s[~is_format & ~is_empty]
        real_format = is_format & ~is_zero
        parsed = pd.to_datetime(txt.where(real_format), format="%Y-%m-%d", errors="coerce")
        cannot = s[real_format & parsed.isna()]
        good, stamp = s[parsed.notna()], parsed[parsed.notna()]
        before_1990 = int(good[stamp < pd.Timestamp("1990-01-01")].sum())
        after_crawl = int(good[stamp > pd.Timestamp("2018-05-25")].sum())
        print(f"--- {label}: {total:,} rows, {len(s):,} distinct texts")
        print(f"  empty {int(s[is_empty].sum()):,} | 0000-00-00 {int(s[is_zero].sum()):,} | other formats {int(other.sum()):,} {other.head(5).to_dict()}")
        print(f"  right format but not a real date (for example 2010-00-00 or 2018-02-31): {int(cannot.sum()):,} {cannot.head(5).to_dict()}")
        if len(good):
            by_md = good.groupby(good.index.str[5:10]).sum().sort_values(ascending=False)
            by_year = good.groupby(stamp.dt.year).sum().sort_values(ascending=False)
            share_jan1 = by_md.get("01-01", 0) / good.sum()
            print(f"  real dates: {int(good.sum()):,} | earliest {stamp.min().date()} | latest {stamp.max().date()}")
            print(f"  before 1990: {before_1990:,} | after the crawl (2018-05-25): {after_crawl:,}")
            print("  most frequent month-day:", by_md.head(5).to_dict(), f"| share of 01-01: {share_jan1:.2%} (evenly spread: 0.27%)")
            print("  most frequent years:", by_year.head(5).to_dict())

    summarize_dates(date_counts["my_start_date"], "my_start_date")
    print()
    summarize_dates(date_counts["my_finish_date"], "my_finish_date")
    print(f"\nfinish date before start date: {finish_before_start:,} of {both_real:,} rows that have two real dates")

    print("\nmy_last_updated (seconds since 1970):")
    print("  earliest:", pd.to_datetime(upd_min, unit="s"), "| latest:", pd.to_datetime(upd_max, unit="s"))
    print(f"  <= 0: {upd_nonpositive:,} | before November 2004 (MyAnimeList did not exist): {upd_before_site:,} | after 2018-05-25: {upd_after_crawl:,}")
else:
    print("Skipped (RUN_LISTS = False)")


--- my_start_date: 80,076,112 rows, 12,536 distinct texts
  empty 0 | 0000-00-00 69,010,299 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 425,434 {'2008-00-00': 25904, '2007-00-00': 22607, '2009-00-00': 24032, '2011-00-00': 19978, '2012-00-00': 20127}
  real dates: 10,640,379 | earliest 1693-09-25 | latest 2115-04-22
  before 1990: 1,762 | after the crawl (2018-05-25): 272
  most frequent month-day: {'04-07': 85889, '04-06': 83280, '01-10': 82223, '04-08': 81944, '04-05': 81766} | share of 01-01: 0.42% (evenly spread: 0.27%)
  most frequent years: {2015: 1392932, 2016: 1374371, 2014: 1330945, 2017: 1306770, 2013: 1190143}

--- my_finish_date: 80,076,112 rows, 12,441 distinct texts
  empty 0 | 0000-00-00 66,550,407 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 351,987 {'2007-00-00': 18480, '2008-00-00': 21644, '2009-00-00': 20172, '2011-00-00': 17397, '2013-00-00': 17083}
  real dates: 13

In [11]:
# B4. anime_id and usernames of the list table
if RUN_LISTS:
    orphan_ids = ids - known_ids
    print(f"anime_id in the list but not in AnimeList: {len(orphan_ids):,} ids, {orphan_rows:,} rows"
          f" | anime without any list row: {len(known_ids - ids):,} of {len(known_ids):,}")

    listed = pd.Series(sorted(names))
    low = listed.str.lower()
    clash = low.duplicated(keep=False)
    print(f"\nusernames in the list table: {len(listed):,} | empty username cells: {empty_username:,}")
    print("distinct after lowercase:", f"{low.nunique():,}", "| names that differ only by capital letters:", int(clash.sum()))
    print("  examples:", listed[clash].head(10).tolist())
    print("  with a space at the start or the end:", int((listed != listed.str.strip()).sum()))

    user_names = set(users["username"])
    lower_users = set(users["username"].str.lower())
    not_found = [n for n in listed if n not in user_names]
    only_case = [n for n in not_found if n.lower() in lower_users]
    print("list usernames not found in UserList (exact text):", len(not_found),
          "| found if capital letters are ignored:", len(only_case), "| examples:", not_found[:5])
else:
    print("Skipped (RUN_LISTS = False)")


anime_id in the list but not in AnimeList: 0 ids, 0 rows | anime without any list row: 0 of 14,478

usernames in the list table: 283,045 | empty username cells: 0
distinct after lowercase: 283,045 | names that differ only by capital letters: 0
  examples: []
  with a space at the start or the end: 0
list usernames not found in UserList (exact text): 0 | found if capital letters are ignored: 0 | examples: []


In [12]:
# B5. my_watched_episodes against the episode count of the anime
if RUN_LISTS:
    print(f"rows whose anime has a known episode count (> 0): {ep_known:,} ({ep_known / rows:.1%})")
    print(f"my_watched_episodes above the episode count of the anime: {watched_over:,}")
    print(f"completed rows: {completed_rows:,} | with my_watched_episodes == 0: {completed_zero:,}"
          f" | below the episode count: {completed_less:,}")
else:
    print("Skipped (RUN_LISTS = False)")


rows whose anime has a known episode count (> 0): 79,375,560 (99.1%)
my_watched_episodes above the episode count of the anime: 207,305
completed rows: 50,445,526 | with my_watched_episodes == 0: 98,519 | below the episode count: 282,454


### Observations for part B

**TODO (fill in after the run):**
- B2: how many `my_tags` are placeholder-like, and which words? Do the most frequent tags look like real tags? How many contain an HTML code?
- B3: how many start and finish dates are `0000-00-00`, partial (`2010-00-00`), impossible, before 1990, or after the crawl? Is January 1 over-represented (compare with 0.27%)? How many finish dates come before the start date? Is `my_last_updated` ever before 2004 or after the crawl?
- B4: are there `anime_id` values that do not exist in `AnimeList`? Do usernames of the list table differ from those of `UserList` only by capital letters or spaces?
- B5: how many rows have more episodes watched than the anime has? How many completed rows have 0 or too few watched episodes?
- Which of these problems is new for the summary of step 3 (to add after the merge)?
